# Phase 4 — Model training and comparison

Train **Naive Bayes** (baseline), **Logistic Regression**, and **linear SVM** (`LinearSVC`) on TF-IDF features. Record **training time** and **accuracy on the held-out test set** (same split and vectorizer as `feature_extraction.ipynb`). The **best** model by test accuracy is saved as **`models/best_model.pkl`** (use with `models/tfidf_vectorizer.pkl` at inference).

In [ ]:
%pip install -q pandas scikit-learn scipy joblib

In [ ]:
from __future__ import annotations

import time
from pathlib import Path

import joblib
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name in {"notebooks", "notebook_runs"} else NOTEBOOK_DIR
DATA_DIR = ROOT / "data"
MODELS_DIR = ROOT / "models"

PROCESSED_CSV = DATA_DIR / "processed_dataset.csv"
VECTORIZER_PATH = MODELS_DIR / "tfidf_vectorizer.pkl"
BEST_MODEL_PATH = MODELS_DIR / "best_model.pkl"

RANDOM_STATE = 42
TEST_SIZE = 0.20

In [ ]:
if not VECTORIZER_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {VECTORIZER_PATH}. Run notebooks/feature_extraction.ipynb first."
    )

df = pd.read_csv(PROCESSED_CSV)
df = df.dropna(subset=["text_clean", "target"])
df["text_clean"] = df["text_clean"].astype(str).str.strip()
df = df[df["text_clean"].str.len() > 0]

X_raw = df["text_clean"]
y = df["target"].astype(int)
if not set(y.unique()).issubset({0, 1, 2}):
    raise ValueError("Expected target in {0,1,2}.")

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

vectorizer = joblib.load(VECTORIZER_PATH)
X_train = vectorizer.transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)

print("Train:", X_train.shape[0], " Test:", X_test.shape[0])
print("Feature matrix:", X_train.shape)

In [ ]:
models_cfg: list[tuple[str, object]] = [
    ("Naive Bayes", MultinomialNB()),
    (
        "Logistic Regression",
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    ),
    (
        "SVM (LinearSVC)",
        LinearSVC(random_state=RANDOM_STATE, max_iter=8000),
    ),
]

results: list[dict] = []
fitted: dict[str, object] = {}

for name, clf in models_cfg:
    t0 = time.perf_counter()
    clf.fit(X_train, y_train)
    train_time_s = time.perf_counter() - t0
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results.append(
        {"model": name, "train_time_s": train_time_s, "validation_accuracy": acc}
    )
    fitted[name] = clf
    print(f"{name}: train_time={train_time_s:.4f}s  validation_acc={acc:.4f}")

summary = pd.DataFrame(results).sort_values(
    "validation_accuracy", ascending=False
)
display(summary)

In [ ]:
best_name = summary.iloc[0]["model"]
best_clf = fitted[best_name]
best_acc = float(summary.iloc[0]["validation_accuracy"])

joblib.dump(best_clf, BEST_MODEL_PATH)
print(f"Best model: {best_name}  (validation accuracy={best_acc:.4f})")
print("Saved:", BEST_MODEL_PATH)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# Use the saved best model for final evaluation on the test split.
model = best_clf

# Predict on test data
y_pred = model.predict(X_test)

# Generate classification report
report = classification_report(y_test, y_pred)

# Print report
print(report)

# Save report to file
with open(DATA_DIR / "classification_report.txt", "w", encoding="utf-8") as f:
    f.write(report)

# Calculate accuracy separately
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

In [ ]:
REALISTIC_BENCHMARK_CSV = DATA_DIR / "realistic_benchmark.csv"

if REALISTIC_BENCHMARK_CSV.is_file():
    benchmark_df = pd.read_csv(REALISTIC_BENCHMARK_CSV)
    benchmark_X = vectorizer.transform(benchmark_df["text"].astype(str))
    benchmark_y = benchmark_df["target"].astype(int)
    benchmark_pred = best_clf.predict(benchmark_X)

    print("Realistic benchmark classification report:\n")
    print(classification_report(benchmark_y, benchmark_pred, digits=4))
    print("Realistic benchmark accuracy:", accuracy_score(benchmark_y, benchmark_pred))
else:
    print(f"Missing {REALISTIC_BENCHMARK_CSV}. Skipping realistic benchmark evaluation.")